# Fusion Retrieval: Combining Vector Search + BM25 Keyword Search

## Overview

Vector search is great at **semantic matching** but can miss exact keywords. BM25 is great at **keyword matching** but doesn't understand meaning. **Fusion retrieval** combines both to get the best of both worlds.

| Method | Strengths | Weaknesses |
|---|---|---|
| **Vector search** (FAISS) | Understands meaning, synonyms, paraphrases | May miss exact keyword matches |
| **BM25** (keyword) | Excellent at exact term matching | No semantic understanding |
| **Fusion** | Both meaning AND keywords | Slightly more complex |

## How It Works

1. Run **both** vector search and BM25 search on the same query
2. Normalize both sets of scores to [0, 1]
3. Combine with a weighted average: `alpha * vector_score + (1-alpha) * bm25_score`
4. Rank by combined score, return top-k

The `alpha` parameter controls the balance: `alpha=1.0` = pure vector, `alpha=0.0` = pure BM25, `alpha=0.5` = equal weight.

## Models Used

- **Embeddings**: `mxbai-embed-large:335m` via Ollama
- **BM25**: `rank_bm25` library (no model needed, pure statistical)

<div style="text-align: center;">

<img src="./images/fusion_retrieval.svg" alt="Fusion Retrieval" style="width:100%; height:auto;">
</div>

---
## Step 0: Import Packages

In [1]:
import numpy as np
from rank_bm25 import BM25Okapi
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama.embeddings import OllamaEmbeddings
from IPython.display import display, HTML

---
## Step 1: Set Up the Embedding Model

In [2]:
embedding_model = OllamaEmbeddings(model="mxbai-embed-large:335m")

print("Embedding model ready")

Embedding model ready


---
## Step 2: Load PDF, Chunk, and Build the Vector Store

In [3]:
path = "data/Understanding_Climate_Change.pdf"

loader = PyPDFLoader(path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, length_function=len
)
chunks = text_splitter.split_documents(documents)

# Clean chunks
for chunk in chunks:
    chunk.page_content = chunk.page_content.replace("\t", " ")

vectorstore = FAISS.from_documents(chunks, embedding_model)

print(f"Loaded {len(documents)} pages, split into {len(chunks)} chunks")
print(f"FAISS vector store created")

Loaded 33 pages, split into 97 chunks
FAISS vector store created


---
## Step 3: Build the BM25 Index

BM25 (Best Matching 25) is a classic keyword-based ranking function. It scores documents by how often the query terms appear, adjusted for document length. We build the BM25 index from the **same chunks** used in the vector store.

In [4]:
tokenized_docs = [chunk.page_content.split() for chunk in chunks]
bm25 = BM25Okapi(tokenized_docs)

print(f"BM25 index created from {len(tokenized_docs)} chunks")

BM25 index created from 97 chunks


---
## Step 4: Perform Fusion Retrieval

This is the core logic — inline, step by step:

1. Run BM25 scoring on the query
2. Run vector similarity search on the query
3. Normalize both score sets to [0, 1]
4. Combine with weighted average
5. Rank by combined score

In [5]:
query = "What are the impacts of climate change on the environment?"
k = 5
alpha = 0.5  # 0.5 = equal weight to vector and BM25
epsilon = 1e-8  # small value to avoid division by zero

print(f"Query: {query}")
print(f"alpha = {alpha} (0 = pure BM25, 1 = pure vector, 0.5 = equal)\n")

# --- Step A: Get all documents from vectorstore (we need them in order) ---
all_docs = vectorstore.similarity_search("", k=vectorstore.index.ntotal)

# --- Step B: BM25 scores ---
bm25_scores = bm25.get_scores(query.split())
print(f"BM25 scores range: [{bm25_scores.min():.4f}, {bm25_scores.max():.4f}]")

# --- Step C: Vector similarity scores ---
vector_results = vectorstore.similarity_search_with_score(query, k=len(all_docs))
vector_scores = np.array([score for _, score in vector_results])
print(f"Vector scores range: [{vector_scores.min():.4f}, {vector_scores.max():.4f}]")

# --- Step D: Normalize both to [0, 1] ---
# Vector: lower distance = better, so we invert
vector_scores = 1 - (vector_scores - vector_scores.min()) / (vector_scores.max() - vector_scores.min() + epsilon)

# BM25: higher = better, just normalize
bm25_scores = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + epsilon)

print(f"\nNormalized vector scores range: [{vector_scores.min():.4f}, {vector_scores.max():.4f}]")
print(f"Normalized BM25 scores range:   [{bm25_scores.min():.4f}, {bm25_scores.max():.4f}]")

# --- Step E: Combine ---
combined_scores = alpha * vector_scores + (1 - alpha) * bm25_scores

# --- Step F: Rank and get top-k ---
sorted_indices = np.argsort(combined_scores)[::-1]
top_docs = [all_docs[i] for i in sorted_indices[:k]]

print(f"\nTop {k} combined scores: {[f'{combined_scores[i]:.4f}' for i in sorted_indices[:k]]}")

Query: What are the impacts of climate change on the environment?
alpha = 0.5 (0 = pure BM25, 1 = pure vector, 0.5 = equal)

BM25 scores range: [0.8274, 10.4482]
Vector scores range: [0.4868, 1.0798]

Normalized vector scores range: [0.0000, 1.0000]
Normalized BM25 scores range:   [0.0000, 1.0000]

Top 5 combined scores: ['0.9665', '0.9325', '0.9119', '0.8349', '0.8092']


---
## Step 5: Display the Retrieved Documents

In [6]:
for i, doc in enumerate(top_docs):
    display(HTML(f"<span style='background-color: lightblue;'> <b>Result {i}:</b></span>"))
    display(HTML(f"<b>Content:</b> {doc.page_content[:300]}..."))
    display(HTML(f"<b>Source:</b> page {doc.metadata.get('page', 'N/A')}"))
    print("=" * 80)

---
## Step 6 (Optional): Compare Different Alpha Values

Let's see how the top result changes when we shift the balance between vector and BM25.

In [7]:
for test_alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    scores = test_alpha * vector_scores + (1 - test_alpha) * bm25_scores
    best_idx = np.argmax(scores)
    print(f"alpha={test_alpha:.2f}: top result = \"{all_docs[best_idx].page_content[:80]}...\"")

alpha=0.00: top result = "Investing in renewable energy, energy efficiency, and sustainable practices crea..."
alpha=0.25: top result = "partnerships, and developing joint initiatives. Digital tools and online network..."
alpha=0.50: top result = "partnerships, and developing joint initiatives. Digital tools and online network..."
alpha=0.75: top result = "partnerships, and developing joint initiatives. Digital tools and online network..."
alpha=1.00: top result = "arid and semi-arid regions. Droughts can lead to food and water shortages and ex..."


---
## Summary

| Step | What happened |
|---|---|
| 1 | Set up embedding model |
| 2 | Loaded PDF, chunked, built FAISS vector store |
| 3 | Built BM25 keyword index from the same chunks |
| 4 | **Fusion retrieval**: ran both searches, normalized scores, combined with alpha-weighted average |
| 5 | Displayed top-5 results |
| 6 | Compared how different alpha values shift the ranking |

**Key insight:** Fusion retrieval catches documents that either method alone might miss. A query like "CO2 emissions impact" benefits from BM25 (exact keyword "CO2") AND vector search (semantic understanding of "impact" = consequences/effects). The alpha parameter lets you tune the balance per use case.